# Variational Autoencoder for EuroSAT

This notebook implements a VAE for 64x64 RGB satellite images (EuroSAT) and fulfills:
- Encoder, decoder, and reparameterization trick
- VAE loss (reconstruction + KL)
- Training for 15 epochs with loss curve
- Random sample generation
- Latent interpolation
- Semantic vector arithmetic + executive summary


In [ ]:
#If CUDA is available, use the following command to install PyTorch without CUDA support:
%pip install numpy pandas scikit-learn matplotlib seaborn torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124

#If you have an Intel GPU and want to use PyTorch with XPU support, use the following command:
#pip install numpy pandas scikit-learn matplotlib seaborn torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/xpu

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

In [ ]:
data_root = Path("data")
img_size = 64

transform = transforms.Compose(
    [
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ]
)

# EuroSAT is a built-in torchvision dataset. It will download if missing.
dataset = datasets.EuroSAT(root=data_root, download=True, transform=transform)
print("Total images:", len(dataset))
print("Classes:", dataset.classes)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(seed)
)

batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

idx_to_class = {i: c for i, c in enumerate(dataset.classes)}
class_to_idx = {c: i for i, c in enumerate(dataset.classes)}
print("Split sizes:", len(train_set), len(val_set), len(test_set))

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # 64 -> 32
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 32 -> 16
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 16 -> 8
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),  # 8 -> 4
            nn.ReLU(inplace=True),
        )
        self.enc_out_dim = 256 * 4 * 4
        self.fc_mu = nn.Linear(self.enc_out_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.enc_out_dim, latent_dim)

        self.fc_dec = nn.Linear(latent_dim, self.enc_out_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 4 -> 8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 8 -> 16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 16 -> 32
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),  # 32 -> 64
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(x.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(z.size(0), 256, 4, 4)
        x_hat = self.decoder(h)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


model = VAE(latent_dim=32).to(device)
print(model)

In [ ]:
def vae_loss(x_hat, x, mu, logvar):
    recon = F.mse_loss(x_hat, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl, recon, kl

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 100
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    # ── Train ──────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        x_hat, mu, logvar = model(x)
        loss, recon, kl = vae_loss(x_hat, x, mu, logvar)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_train = running_loss / len(train_loader.dataset)
    train_losses.append(avg_train)

    # ── Validation ─────────────────────────────────────
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for x, _ in val_loader:
            x = x.to(device)
            x_hat, mu, logvar = model(x)
            loss, _, _ = vae_loss(x_hat, x, mu, logvar)
            val_running += loss.item()
    avg_val = val_running / len(val_loader.dataset)
    val_losses.append(avg_val)

    print(f"Epoch {epoch:03d} | train loss: {avg_train:.4f} | val loss: {avg_val:.4f}")


In [ ]:
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses,   label="Val Loss")
plt.title("Train vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (per sample)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Final  train loss : {train_losses[-1]:.4f}")
print(f"Final  val loss   : {val_losses[-1]:.4f}")


In [ ]:
latent_temperature = 0.2

model.eval()
with torch.no_grad():
    z = torch.randn(16, model.latent_dim, device=device) * latent_temperature
    samples = model.decode(z).cpu()

grid = make_grid(samples, nrow=4)
plt.figure(figsize=(6, 6))
plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
plt.title("Random Samples")
plt.axis("off")
plt.show()


In [ ]:
def find_first_by_class(ds, target_class_name):
    target_idx = class_to_idx[target_class_name]
    for img, label in ds:
        if label == target_idx:
            return img, label
    raise ValueError(f"Class not found: {target_class_name}")

img_a, label_a = find_first_by_class(test_set, "Industrial")
img_b, label_b = find_first_by_class(test_set, "Pasture")

model.eval()
with torch.no_grad():
    x_a = img_a.unsqueeze(0).to(device)
    x_b = img_b.unsqueeze(0).to(device)
    mu_a, logvar_a = model.encode(x_a)
    mu_b, logvar_b = model.encode(x_b)
    z_a = mu_a
    z_b = mu_b

    steps = 10
    alphas = torch.linspace(0, 1, steps, device=device)
    z_interp = torch.stack([(1 - a) * z_a + a * z_b for a in alphas]).squeeze(1)
    imgs_interp = model.decode(z_interp).cpu()

plt.figure(figsize=(12, 2))
for i in range(steps):
    ax = plt.subplot(1, steps, i + 1)
    ax.imshow(np.transpose(imgs_interp[i].numpy(), (1, 2, 0)))
    ax.axis("off")
plt.suptitle("Latent Space Interpolation: Industrial -> Pasture")
plt.show()

In [ ]:
def collect_latent_means(ds, target_class_name, max_count=128):
    target_idx = class_to_idx[target_class_name]
    latents = []
    model.eval()
    with torch.no_grad():
        for img, label in ds:
            if label == target_idx:
                x = img.unsqueeze(0).to(device)
                mu, _ = model.encode(x)
                latents.append(mu.squeeze(0))
                if len(latents) >= max_count:
                    break
    if not latents:
        raise ValueError(f"No samples for class {target_class_name}")
    return torch.stack(latents, dim=0).mean(dim=0)

forest_mean = collect_latent_means(train_set, "Forest", max_count=128)
highway_mean = collect_latent_means(train_set, "Highway", max_count=128)

paving_direction = highway_mean - forest_mean

river_img, _ = find_first_by_class(test_set, "River")

model.eval()
with torch.no_grad():
    river_x = river_img.unsqueeze(0).to(device)
    river_mu, _ = model.encode(river_x)
    river_mod = river_mu + paving_direction.unsqueeze(0)

    river_recon = model.decode(river_mu).cpu().squeeze(0)
    river_paved = model.decode(river_mod).cpu().squeeze(0)

plt.figure(figsize=(6, 3))
ax1 = plt.subplot(1, 2, 1)
ax1.imshow(np.transpose(river_recon.numpy(), (1, 2, 0)))
ax1.set_title("River (recon)")
ax1.axis("off")

ax2 = plt.subplot(1, 2, 2)
ax2.imshow(np.transpose(river_paved.numpy(), (1, 2, 0)))
ax2.set_title("River + paving")
ax2.axis("off")

plt.suptitle("Semantic Vector Arithmetic")
plt.show()

## Executive Summary (max 300 words)

The semantic vector arithmetic experiment suggests the VAE has learned a weak but meaningful notion of land-use texture. After computing the average latent for Forest and Highway and applying the resulting paving direction to a River image, the decoded output shows subtle changes in texture: the river region becomes more linear and less organic, with faint grid-like or elongated patterns that resemble paved or channelized surfaces. The overall river geometry remains recognizable, indicating that the latent edit mainly influences surface texture rather than global shape.

This outcome implies that the model captures some disentangled factors: a texture axis related to paved infrastructure can be transferred across classes, even though the effect is not perfectly isolated. The incomplete transformation (a river that still looks partly like water) is expected for a compact 32-dim latent space and a relatively small number of training epochs. Increasing training time, model capacity, or using a beta-VAE style weighting could strengthen this semantic direction and reduce entanglement.

Overall, the results indicate the VAE has internalized class-dependent visual patterns (vegetation vs. built surfaces) and can apply them in a controlled way. The experiment supports the claim that the latent space encodes semantic structure in addition to low-level pixel statistics, but the edits are modest and would benefit from additional training and tuning.